# Bangla Hallucination Detection - v3.0 (Robust)

- **LayerNorm fix**: BanglaBERT weights load correctly (gamma/beta -> weight/bias)
- **5-fold fine-tuning ensemble**: robust OOF predictions
- **XGBoost on [CLS] + NLI features**: strong meta-classifier, no overfitting
- **Blended ensemble**: optimal FT + XGB + NLI weights via grid search
- **No pseudo-labeling**: it backfired in prior version
- **No aggressive augmentation**: only safe character-level noise for TTA

In [ ]:
!pip install -q transformers datasets evaluate accelerate xgboost
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
import json, gc, os, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from normalizer import normalize
from transformers import (
    AutoTokenizer, ElectraForSequenceClassification, ElectraConfig, ElectraModel,
    Trainer, TrainingArguments, EarlyStoppingCallback, pipeline
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from datasets import Dataset
from tqdm.auto import tqdm
import xgboost as xgb
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DATA_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SUBMISSION_PATH = "/kaggle/working/submission_v3.0.csv"
MODEL_NAME = "csebuetnlp/banglabert"
NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Load & Normalize

In [ ]:
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../bengali-hallucination/dataset samples.json"
    TEST_PATH = "../bengali-hallucination/test set.csv"

with open(DATA_PATH) as f:
    records = json.load(f)
df = pd.DataFrame(records)
print(f"Train: {len(df)}")

NO_CTX = {"", "nan", "NaN", "[NULL]", None}
def clean(text):
    if pd.isna(text) or str(text).strip() in NO_CTX:
        return ""
    return normalize(str(text).strip())

for col in ['context', 'prompt_bn', 'response_bn']:
    df[f'clean_{col.split("_")[0]}'] = df[col].apply(clean)
df['has_context'] = df['clean_context'].str.len() > 0

def make_input(row):
    parts = [p for p in [row.clean_context, row.clean_prompt, row.clean_response] if p]
    return ' [SEP] '.join(parts)

df['input_text'] = df.apply(make_input, axis=1)
print(f"Context: {df['has_context'].sum()}, No context: {(~df['has_context']).sum()}")
print(f"Label 0: {(df['label']==0).sum()}, Label 1: {(df['label']==1).sum()}")

## 2. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 256
def tok_fn(ex):
    return tokenizer(ex['input_text'], padding='max_length', truncation=True, max_length=MAX_LEN)

## 3. NLI Entailment Scores

In [ ]:
print("Loading NLI...")
nli = pipeline('zero-shot-classification', model=NLI_MODEL,
               device=0 if device == 'cuda' else -1,
               torch_dtype=torch.float16 if device == 'cuda' else torch.float32)

def nli_scores(df_data):
    scores = np.full(len(df_data), 0.5)
    for i, (_, row) in tqdm(enumerate(df_data.iterrows()), total=len(df_data), desc='NLI'):
        prem = row.clean_context if row.clean_context else row.clean_prompt
        if not prem:
            continue
        try:
            r = nli(prem, candidate_labels=[row.clean_response], hypothesis_template='{}')
            scores[i] = r['scores'][0]
        except:
            pass
    return scores

print("Train NLI...")
df['nli_score'] = nli_scores(df)
print(f"NLI: mean={df['nli_score'].mean():.3f}, std={df['nli_score'].std():.3f}")

## 4. Fixed BanglaBERT

In [ ]:
def make_model(num_labels=2):
    config = ElectraConfig.from_pretrained(MODEL_NAME, num_labels=num_labels)
    model = ElectraForSequenceClassification(config)
    base = ElectraModel.from_pretrained(MODEL_NAME)
    sd = {'electra.' + k: v for k, v in base.state_dict().items()}
    miss, unexp = model.load_state_dict(sd, strict=False)
    cls_miss = [k for k in miss if k.startswith('classifier.')]
    other = [k for k in miss if not k.startswith('classifier.')]
    if other:
        print(f'Other missing: {other}')
    print(f'Classifier head ({len(cls_miss)} keys) randomly initialized.')
    return model

## 5. Metrics & Training

In [ ]:
n0, n1 = (df['label']==0).sum(), (df['label']==1).sum()
cw = torch.tensor([len(df)/(2*n0), len(df)/(2*n1)], dtype=torch.float32)
print(f'Class weights: 0={cw[0]:.3f}, 1={cw[1]:.3f}')

class CW_Trainer(Trainer):
    def __init__(self, cw=None, **kw):
        super().__init__(**kw)
        self.cw = cw
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits, labels,
                               weight=self.cw.to(labels.device)) if self.cw is not None else F.cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def metrics(ep):
    p = np.argmax(ep.predictions, -1)
    return {'acc': accuracy_score(ep.label_ids, p), 'macro_f1': f1_score(ep.label_ids, p, average='macro')}

## 6. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(df, df['label'])):
    df.loc[val_idx, 'fold'] = fold

def train_fold(fold_num):
    print(f"\n{'='*40}\nFold {fold_num}\n{'='*40}")
    tr = df[df['fold'] != fold_num].copy()
    va = df[df['fold'] == fold_num].copy()
    print(f"Train: {len(tr)}, Val: {len(va)}")

    tr_ds = Dataset.from_pandas(tr[['input_text','label']]).map(tok_fn, batched=True).remove_columns(['input_text'])
    va_ds = Dataset.from_pandas(va[['input_text','label']]).map(tok_fn, batched=True).remove_columns(['input_text'])

    model = make_model(2).to(device)
    args = TrainingArguments(
        output_dir=f'./f{fold_num}', eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        learning_rate=2e-5, per_device_train_batch_size=8, per_device_eval_batch_size=16,
        num_train_epochs=10, weight_decay=0.01, warmup_ratio=0.1,
        load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
        logging_steps=10, report_to='none', fp16=device == 'cuda',
    )
    trainer = CW_Trainer(cw=cw, model=model, args=args,
                         train_dataset=tr_ds, eval_dataset=va_ds,
                         compute_metrics=metrics,
                         callbacks=[EarlyStoppingCallback(3)])
    trainer.train()

    preds = trainer.predict(va_ds)
    probs = F.softmax(torch.from_numpy(preds.predictions), -1).numpy()
    acc = accuracy_score(va['label'], np.argmax(preds.predictions, -1))
    f1 = f1_score(va['label'], np.argmax(preds.predictions, -1), average='macro')
    print(f"Fold {fold_num}: Acc={acc:.4f}, Macro F1={f1:.4f}")
    return trainer, probs

trainers = []
oof_ft = np.zeros((len(df), 2))
for fold in range(5):
    tr, probs = train_fold(fold)
    trainers.append(tr)
    mask = df['fold'] == fold
    oof_ft[mask] = probs
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

p = np.argmax(oof_ft, 1)
print(f"\nFT OOF: Acc={accuracy_score(df['label'], p):.4f}, Macro F1={f1_score(df['label'], p, average='macro'):.4f}")
print(confusion_matrix(df['label'], p))

## 7. XGBoost on [CLS] + NLI

In [ ]:
def get_cls(texts, model, batch_size=16):
    model.eval()
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='CLS'):
        enc = tokenizer(texts[i:i+batch_size], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.concatenate(embs)

# Use fold-0 model's electra for embeddings
emb_model = make_model(2).to(device)
print("Extracting [CLS]...")
train_cls = get_cls(df['input_text'].tolist(), emb_model.electra)
print(f"Shape: {train_cls.shape}")

# Features: [CLS] + NLI
X = np.column_stack([train_cls, df['nli_score'].values])
y = df['label'].values

print("XGBoost 5-fold...")
xgb_oof = np.zeros(len(df))
skf2 = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr, va) in enumerate(skf2.split(X, y)):
    m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=2,
                          scale_pos_weight=(n0/n1), random_state=RANDOM_STATE,
                          eval_metric='logloss', early_stopping_rounds=30, use_label_encoder=False, verbosity=0)
    m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
    xgb_oof[va] = m.predict_proba(X[va])[:, 1]

xgb_p = (xgb_oof >= 0.5).astype(int)
print(f"XGB OOF: Acc={accuracy_score(y, xgb_p):.4f}, Macro F1={f1_score(y, xgb_p, average='macro'):.4f}")
print(confusion_matrix(y, xgb_p))

## 8. Blend Fine-Tune + XGBoost + NLI
Grid search optimal weights on OOF. Only safe ensemble, no pseudo-labeling.

In [ ]:
ft_p = oof_ft[:, 1]
xgb_p = xgb_oof
nli_p = df['nli_score'].values

best_f1, best_w = 0, None
for w1 in np.arange(0, 1.05, 0.1):
    for w2 in np.arange(0, 1.05, 0.1):
        w3 = 1 - w1 - w2
        if w3 < -0.001:
            continue
        blended = w1 * ft_p + w2 * xgb_p + w3 * nli_p
        p = (blended >= 0.5).astype(int)
        f1 = f1_score(y, p, average='macro')
        if f1 > best_f1:
            best_f1, best_w = f1, (w1, w2, w3)

w1, w2, w3 = best_w
print(f"Optimal weights: FT={w1:.1f}, XGB={w2:.1f}, NLI={w3:.1f}")
print(f"Blended OOF Macro F1: {best_f1:.4f}")

final_blend = w1 * ft_p + w2 * xgb_p + w3 * nli_p
final_p = (final_blend >= 0.5).astype(int)
print(f"Blend distribution: 0={(final_p==0).sum()}, 1={(final_p==1).sum()}")
print(confusion_matrix(y, final_p))

## 9. Test Set Predictions

In [ ]:
test_df = pd.read_csv(TEST_PATH)
print(f"Test: {len(test_df)}")
for col in ['context', 'prompt_bn', 'response_bn']:
    test_df[f'clean_{col.split("_")[0]}'] = test_df[col].apply(clean)
test_df['has_context'] = test_df['clean_context'].str.len() > 0
test_df['input_text'] = test_df.apply(make_input, axis=1)

# NLI
print("Test NLI...")
test_df['nli_score'] = nli_scores(test_df)

# Fine-tuned predictions (avg across 5 folds)
print("Test FT predictions...")
ft_test = []
for fold, tr in enumerate(trainers):
    ds = Dataset.from_pandas(test_df[['input_text']]).map(tok_fn, batched=True).remove_columns(['input_text'])
    preds = tr.predict(ds)
    ft_test.append(F.softmax(torch.from_numpy(preds.predictions), -1).numpy()[:, 1])
ft_test = np.mean(ft_test, 0)

# XGBoost (refit on full data)
print("Test XGBoost...")
test_cls = get_cls(test_df['input_text'].tolist(), emb_model.electra)
X_test = np.column_stack([test_cls, test_df['nli_score'].values])
xgb_full = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, min_child_weight=2,
                              scale_pos_weight=(n0/n1), random_state=RANDOM_STATE,
                              eval_metric='logloss', use_label_encoder=False, verbosity=0)
xgb_full.fit(X, y)
xgb_test = xgb_full.predict_proba(X_test)[:, 1]

# Blend
test_blend = w1 * ft_test + w2 * xgb_test + w3 * test_df['nli_score'].values
test_preds = (test_blend >= 0.5).astype(int)

n0 = (test_preds==0).sum()
n1 = (test_preds==1).sum()
print(f"\nTest distribution: 0={n0} ({n0/len(test_preds)*100:.1f}%), 1={n1} ({n1/len(test_preds)*100:.1f}%)")

## 10. Save

In [ ]:
sub = pd.DataFrame({'id': test_df['id'], 'label': test_preds})
os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved: {SUBMISSION_PATH}")
print(f"0: {(test_preds==0).sum()}, 1: {(test_preds==1).sum()}")